# Part 1. Intersections of right lifting classes of morphisms whose codomains are transitive.

There are 6 *primordial classes*, $\mathsf{inj}^e, \mathsf{inj}^{C_p}, \mathsf{surj}^e, \mathsf{surj}^{C_p}, \mathsf{pb},$ and $\mathsf{homog}$. We implement these as an enum type, `PrimordialClass`. `PRIMORDIAL_IMPLICATION_RULES` records pairs $(S,T)$ where $S$ and $T$ are sets of primordial classes such that $S \nsupseteq T$ and $\bigcap S \subseteq \bigcap T$. We will check later that every containment between intersections of primordial classes is a formal consequence of those recorded in `PRIMORDIAL_IMPLICATION_RULES`.


In [65]:
from dataclasses import dataclass, field
from enum import Enum
from itertools import combinations

from sage.all import JoinSemilattice, LatticePoset, MeetSemilattice, Poset


class PrimordialClass(Enum):
    """One of the six primordial classes."""

    INJ_E = (0, "injᵉ", r"\inj^e")
    INJ_CP = (1, "inj^{Cₚ}", r"\inj^{C_p}")
    SURJ_E = (2, "surjᵉ", r"\surj^e")
    SURJ_CP = (3, "surj^{Cₚ}", r"\surj^{C_p}")
    PB = (4, "pb", r"\pb")
    HOMOG = (5, "homog", r"\homog")

    @property
    def sort_key(self):
        return self.value[0]

    @property
    def label(self):
        return self.value[1]

    @property
    def latex(self):
        return self.value[2]

    def __str__(self):
        return self.label

    def __repr__(self):
        return self.label


# If every class on the left occurs in an intersection, the classes on the
# right are implied. Equivalently, the left intersection is contained in the
# right intersection.
PRIMORDIAL_IMPLICATION_RULES = (
    (
        {PrimordialClass.INJ_E},
        {PrimordialClass.INJ_CP, PrimordialClass.PB, PrimordialClass.HOMOG},
    ),
    ({PrimordialClass.PB}, {PrimordialClass.HOMOG}),
    (
        {PrimordialClass.INJ_E, PrimordialClass.SURJ_E},
        {
            PrimordialClass.INJ_CP,
            PrimordialClass.SURJ_CP,
            PrimordialClass.PB,
            PrimordialClass.HOMOG,
        },
    ),
    (
        {PrimordialClass.SURJ_E, PrimordialClass.PB},
        {PrimordialClass.SURJ_CP},
    ),
    (
        {PrimordialClass.SURJ_CP, PrimordialClass.HOMOG},
        {PrimordialClass.PB},
    ),
)


We implement intersections of primordial classes (which we call *primordial intersections*) with a new type, `PrimordialIntersection`. Each `PrimordialIntersection` stores a set of primordial classes $S$ and represents $\bigcap S$. We enforce that the stored set $S$ (the field `factors`) is maximal: $\bigcap S' \neq \bigcap S$ for all $S' \supsetneq S$. Consequently:

- equality (`==`) of primordial intersections is equality of their `factors` fields;
- containment (`<=`) of primordial intersections is reverse containment of their `factors` fields.


In [66]:
@dataclass(frozen=True)
class PrimordialIntersection:
    """An intersection stored as its maximal, implication-closed set of factors."""

    factors: frozenset = field(default_factory=frozenset)

    def __post_init__(self):
        closure = set(self.factors)
        if not closure.issubset(set(PrimordialClass)):
            raise TypeError("factors must be primordial classes")

        changed = True
        while changed:
            changed = False
            for hypotheses, consequences in PRIMORDIAL_IMPLICATION_RULES:
                if hypotheses <= closure and not consequences <= closure:
                    closure.update(consequences)
                    changed = True

        object.__setattr__(self, "factors", frozenset(closure))

    def _ordered_factors(self):
        """Return the factors in a reproducible order."""
        return tuple(sorted(self.factors, key=lambda factor: factor.sort_key))

    @property
    def sort_key(self):
        ordered_factors = self._ordered_factors()
        return (len(ordered_factors), tuple(factor.sort_key for factor in ordered_factors))

    def intersection(self, other):
        return type(self)(self.factors | other.factors)

    def __le__(self, other):
        if not isinstance(other, PrimordialIntersection):
            return NotImplemented
        # More factors mean a smaller intersection.
        return other.factors <= self.factors

    def __lt__(self, other):
        if not isinstance(other, PrimordialIntersection):
            return NotImplemented
        return self != other and other.factors <= self.factors

    def _display_factors(self):
        """Choose a deterministic minimal generating set for display."""
        ordered_factors = self._ordered_factors()
        for size in range(len(ordered_factors) + 1):
            for candidate in combinations(ordered_factors, size):
                if type(self)(candidate).factors == self.factors:
                    return candidate
        raise AssertionError("the canonical factors must generate themselves")

    def __str__(self):
        display_factors = self._display_factors()
        if not display_factors:
            return "all"
        return " ∩ ".join(factor.label for factor in display_factors)

    def __repr__(self):
        return str(self)

    def _latex_(self):
        display_factors = self._display_factors()
        if not display_factors:
            return r"\all"
        return r" \cap ".join(factor.latex for factor in display_factors)


We now compute the meet-semilattice of primordial intersections: the closure of the primordial classes under intersection, together with the containment order. This amounts to producing `PrimordialIntersection(S)` for every subset $S$ of the primordial classes and ordering the resulting distinct values by containment.

The collection of all primordial intersections is the tuple `primordial_intersections`, and `primordial_intersection_semilattice` is the ambient Sage meet-semilattice.


In [67]:
primordial_intersections = tuple(
    sorted(
        {
            PrimordialIntersection(factors)
            for size in range(len(PrimordialClass) + 1)
            for factors in combinations(PrimordialClass, size)
        },
        key=lambda intersection: intersection.sort_key,
    )
)

primordial_intersection_semilattice = MeetSemilattice(
    (primordial_intersections, lambda left, right: left <= right),
    facade=True,
)

The five *primordial vacuous classes* arising from morphisms with transitive codomain are represented by `PrimordialVacuousClass`. Their poset is not closed under unions: adjoining the union $\mathsf{init}^e \cup \{\operatorname{id}_\varnothing\}^{C_p}$ produces the six *collateral classes*. A `CollateralClass` stores a canonical minimal set of primordial vacuous generators for such a union, and `collateral_semilattice` is their join-semilattice under union.

In [68]:
class PrimordialVacuousClass(Enum):
    """A vacuous RLC arising from a morphism with transitive codomain."""

    ID_EMPTY = (0, "{id_∅}", r"\{\id_\varnothing\}")
    INIT_E = (1, "initᵉ", r"\init^e")
    ID_EMPTY_CP = (2, "{id_∅}^{Cₚ}", r"\{\id_\varnothing\}^{C_p}")
    INIT_CP_INTERSECT_PB = (
        3,
        "init^{Cₚ} ∩ pb",
        r"\init^{C_p} \cap \pb",
    )
    INIT_CP = (4, "init^{Cₚ}", r"\init^{C_p}")

    @property
    def sort_key(self):
        return self.value[0]

    @property
    def label(self):
        return self.value[1]

    @property
    def latex(self):
        return self.value[2]

    def __str__(self):
        return self.label

    def __repr__(self):
        return self.label


PRIMORDIAL_VACUOUS_COVER_RELATIONS = (
    (PrimordialVacuousClass.ID_EMPTY, PrimordialVacuousClass.INIT_E),
    (PrimordialVacuousClass.ID_EMPTY, PrimordialVacuousClass.ID_EMPTY_CP),
    (
        PrimordialVacuousClass.INIT_E,
        PrimordialVacuousClass.INIT_CP_INTERSECT_PB,
    ),
    (
        PrimordialVacuousClass.ID_EMPTY_CP,
        PrimordialVacuousClass.INIT_CP_INTERSECT_PB,
    ),
    (
        PrimordialVacuousClass.INIT_CP_INTERSECT_PB,
        PrimordialVacuousClass.INIT_CP,
    ),
)

primordial_vacuous_poset = Poset(
    (tuple(PrimordialVacuousClass), PRIMORDIAL_VACUOUS_COVER_RELATIONS),
    cover_relations=True,
    facade=True,
)


@dataclass(frozen=True)
class CollateralClass:
    """A union stored by its minimal primordial-vacuous generators."""

    generators: frozenset = field(default_factory=frozenset)

    def __post_init__(self):
        generators = set(self.generators)
        if not generators.issubset(set(PrimordialVacuousClass)):
            raise TypeError(
                "generators must be primordial vacuous classes"
            )
        if not generators:
            generators.add(PrimordialVacuousClass.ID_EMPTY)

        generators = {
            generator
            for generator in generators
            if not any(
                generator != other
                and primordial_vacuous_poset.le(generator, other)
                for other in generators
            )
        }
        object.__setattr__(self, "generators", frozenset(generators))

    def _ordered_generators(self):
        return tuple(sorted(
            self.generators,
            key=lambda generator: generator.sort_key,
        ))

    @property
    def sort_key(self):
        return sum(1 << generator.sort_key for generator in self.generators)

    def union(self, other):
        if not isinstance(other, CollateralClass):
            return NotImplemented
        return type(self)(self.generators | other.generators)

    def __le__(self, other):
        if not isinstance(other, CollateralClass):
            return NotImplemented
        return all(
            any(
                primordial_vacuous_poset.le(generator, other_generator)
                for other_generator in other.generators
            )
            for generator in self.generators
        )

    def __lt__(self, other):
        if not isinstance(other, CollateralClass):
            return NotImplemented
        return self != other and self <= other

    def __str__(self):
        return " ∪ ".join(
            generator.label for generator in self._ordered_generators()
        )

    def __repr__(self):
        return str(self)

    def _latex_(self):
        return r" \cup ".join(
            generator.latex for generator in self._ordered_generators()
        )

def primordial_vacuous_is_contained_in_collateral_class(
    primordial_vacuous_class,
    collateral_class,
):
    return any(
        primordial_vacuous_poset.le(
            primordial_vacuous_class,
            generator,
        )
        for generator in collateral_class.generators
    )

collateral_classes = tuple(sorted(
    {
        CollateralClass(generators)
        for size in range(len(PrimordialVacuousClass) + 1)
        for generators in combinations(PrimordialVacuousClass, size)
    },
    key=lambda collateral_class: collateral_class.sort_key,
))

collateral_semilattice = JoinSemilattice(
    (collateral_classes, lambda left, right: left <= right),
    facade=True,
)

At this point, we must check that both semilattices are mathematically correct. The containments used to construct them follow from the defining properties of the classes, so it remains to verify every non-containment recorded by their Sage orders.

> For each pair of classes $P$ and $Q$ for which a semilattice records $P \nleq Q$, we must produce a morphism in $P \setminus Q$.

For this, Tables 8 and 9 list the same collection of example morphisms. Table 8 records the primordial classes containing each example, while Table 9 records the collateral classes containing it. In code, it suffices to store membership in the five primordial vacuous classes; membership in every collateral union is then derived.

<span style="color:red">The collection currently has sixteen examples and can be extended if further witnesses are needed. In particular, it includes the map $C_p/e \to C_p/e \amalg C_p/C_p$, which distinguishes $\mathsf{init}^{C_p} \cap \mathsf{pb}$ from $\mathsf{init}^e \cup \{\operatorname{id}_\varnothing\}^{C_p}$.</span>

We then check that the examples witness all required non-containments in both semilattices.

In [69]:
@dataclass(frozen=True)
class ExampleMorphism:
    """An example morphism and the named classes containing it."""

    name: str
    primordial_classes: frozenset
    primordial_vacuous_classes: frozenset


# Tables 8 and 9 use these same rows for primordial and collateral membership.
example_morphisms = (
    ExampleMorphism(
        "∅ → *",
        frozenset({
            PrimordialClass.INJ_E,
            PrimordialClass.INJ_CP,
            PrimordialClass.PB,
            PrimordialClass.HOMOG,
        }),
        frozenset({
            PrimordialVacuousClass.INIT_E,
            PrimordialVacuousClass.INIT_CP_INTERSECT_PB,
            PrimordialVacuousClass.INIT_CP,
        }),
    ),
    ExampleMorphism(
        "* ⨿ * → *",
        frozenset({
            PrimordialClass.SURJ_E,
            PrimordialClass.SURJ_CP,
            PrimordialClass.PB,
            PrimordialClass.HOMOG,
        }),
        frozenset(),
    ),
    ExampleMorphism(
        "Cₚ/e → *",
        frozenset({
            PrimordialClass.INJ_CP,
            PrimordialClass.SURJ_E,
            PrimordialClass.HOMOG,
        }),
        frozenset({
            PrimordialVacuousClass.INIT_CP,
        }),
    ),
    ExampleMorphism(
        "Cₚ/e → Cₚ/e ⨿ Cₚ/e",
        frozenset({
            PrimordialClass.INJ_E,
            PrimordialClass.INJ_CP,
            PrimordialClass.SURJ_CP,
            PrimordialClass.PB,
            PrimordialClass.HOMOG,
        }),
        frozenset({
            PrimordialVacuousClass.ID_EMPTY_CP,
            PrimordialVacuousClass.INIT_CP_INTERSECT_PB,
            PrimordialVacuousClass.INIT_CP,
        }),
    ),
    ExampleMorphism(
        "Cₚ/e ⨿ Cₚ/e → Cₚ/e",
        frozenset({
            PrimordialClass.INJ_CP,
            PrimordialClass.SURJ_E,
            PrimordialClass.SURJ_CP,
            PrimordialClass.PB,
            PrimordialClass.HOMOG,
        }),
        frozenset({
            PrimordialVacuousClass.ID_EMPTY_CP,
            PrimordialVacuousClass.INIT_CP_INTERSECT_PB,
            PrimordialVacuousClass.INIT_CP,
        }),
    ),
    ExampleMorphism(
        "Cₚ/e ⨿ * → *",
        frozenset({
            PrimordialClass.INJ_CP,
            PrimordialClass.SURJ_E,
            PrimordialClass.SURJ_CP,
        }),
        frozenset(),
    ),
    ExampleMorphism(
        "Cₚ/e → Cₚ/e ⨿ Cₚ/Cₚ",
        frozenset({
            PrimordialClass.INJ_E,
            PrimordialClass.INJ_CP,
            PrimordialClass.PB,
            PrimordialClass.HOMOG,
        }),
        frozenset({
            PrimordialVacuousClass.INIT_CP_INTERSECT_PB,
            PrimordialVacuousClass.INIT_CP,
        }),
    ),
    ExampleMorphism(
        "Cₚ/e ⨿ Cₚ/e → Cₚ/e ⨿ Cₚ/Cₚ",
        frozenset({
            PrimordialClass.INJ_CP,
            PrimordialClass.PB,
            PrimordialClass.HOMOG,
        }),
        frozenset({
            PrimordialVacuousClass.INIT_CP_INTERSECT_PB,
            PrimordialVacuousClass.INIT_CP,
        }),
    ),
    ExampleMorphism(
        "Cₚ/e → * ⨿ *",
        frozenset({
            PrimordialClass.INJ_CP,
            PrimordialClass.HOMOG,
        }),
        frozenset({
            PrimordialVacuousClass.INIT_CP,
        }),
    ),
    ExampleMorphism(
        "(∅ → *) ⨿ id_*",
        frozenset({
            PrimordialClass.INJ_E,
            PrimordialClass.INJ_CP,
            PrimordialClass.PB,
            PrimordialClass.HOMOG,
        }),
        frozenset(),
    ),
    ExampleMorphism(
        "(Cₚ/e → *) ⨿ id_*",
        frozenset({
            PrimordialClass.INJ_CP,
            PrimordialClass.SURJ_E,
            PrimordialClass.HOMOG,
        }),
        frozenset(),
    ),
    ExampleMorphism(
        "(Cₚ/e → Cₚ/e ⨿ Cₚ/e) ⨿ id_*",
        frozenset({
            PrimordialClass.INJ_E,
            PrimordialClass.INJ_CP,
            PrimordialClass.SURJ_CP,
            PrimordialClass.PB,
            PrimordialClass.HOMOG,
        }),
        frozenset(),
    ),
    ExampleMorphism(
        "(Cₚ/e ⨿ Cₚ/e → Cₚ/e) ⨿ id_*",
        frozenset({
            PrimordialClass.INJ_CP,
            PrimordialClass.SURJ_E,
            PrimordialClass.SURJ_CP,
            PrimordialClass.PB,
            PrimordialClass.HOMOG,
        }),
        frozenset(),
    ),
    ExampleMorphism(
        "∅ → Cₚ/e",
        frozenset({
            PrimordialClass.INJ_E,
            PrimordialClass.INJ_CP,
            PrimordialClass.SURJ_CP,
            PrimordialClass.PB,
            PrimordialClass.HOMOG,
        }),
        frozenset({
            PrimordialVacuousClass.INIT_E,
            PrimordialVacuousClass.ID_EMPTY_CP,
            PrimordialVacuousClass.INIT_CP_INTERSECT_PB,
            PrimordialVacuousClass.INIT_CP,
        }),
    ),
    ExampleMorphism(
        "id_{Cₚ/e}",
        frozenset(PrimordialClass),
        frozenset({
            PrimordialVacuousClass.ID_EMPTY_CP,
            PrimordialVacuousClass.INIT_CP_INTERSECT_PB,
            PrimordialVacuousClass.INIT_CP,
        }),
    ),
    ExampleMorphism(
        (
            "Cₚ/e ⨿ Cₚ/e → Cₚ/e ⨿ Cₚ/e "
            "(both orbits to the first summand)"
        ),
        frozenset({
            PrimordialClass.INJ_CP,
            PrimordialClass.SURJ_CP,
            PrimordialClass.PB,
            PrimordialClass.HOMOG,
        }),
        frozenset({
            PrimordialVacuousClass.ID_EMPTY_CP,
            PrimordialVacuousClass.INIT_CP_INTERSECT_PB,
            PrimordialVacuousClass.INIT_CP,
        }),
    ),
)

def example_lies_in_primordial_class(example, primordial_class):
    return primordial_class in example.primordial_classes

def example_lies_in_primordial_intersection(example, primordial_intersection):
    return primordial_intersection.factors <= example.primordial_classes


def example_lies_in_primordial_vacuous_class(
    example,
    primordial_vacuous_class,
):
    return (
        primordial_vacuous_class
        in example.primordial_vacuous_classes
    )


def example_lies_in_collateral_class(example, collateral_class):
    return any(
        example_lies_in_primordial_vacuous_class(example, generator)
        for generator in collateral_class.generators
    )


def find_noncontainment_witness(left, right, membership_test_left, membership_test_right):
    """Return an example in left but not right, proving left ⊈ right."""
    for example in example_morphisms:
        if membership_test_left(example, left) and not membership_test_right(example, right):
            return example
    raise AssertionError(f"no example witnesses {left} ⊈ {right}")


# Verify every non-containment P ⊈ Q in the primordial-intersection semilattice.
primordial_noncontainment_witnesses = {
    (left, right): find_noncontainment_witness(
        left,
        right,
        example_lies_in_primordial_intersection,
        example_lies_in_primordial_intersection
    )
    for left in primordial_intersections
    for right in primordial_intersections
    if not primordial_intersection_semilattice.le(left, right)
}

# Verify every non-containment P ⊈ Q in the collateral join-semilattice.
collateral_noncontainment_witnesses = {
    (left, right): find_noncontainment_witness(
        left,
        right,
        example_lies_in_collateral_class,
        example_lies_in_collateral_class
    )
    for left in collateral_classes
    for right in collateral_classes
    if not collateral_semilattice.le(left, right)
}


The ordered tuple `transitive_morphism_types` records the nine cases in Tables 2 and 3. Each immutable `TransitiveMorphismType` has a readable case name, its right-lifting class, and its vacuous right-lifting class.

In [70]:
@dataclass(frozen=True)
class TransitiveMorphismType:
    """One row of the transitive-codomain morphism classification."""

    name: str
    rlc: PrimordialIntersection
    vacuous_rlc: PrimordialVacuousClass


transitive_morphism_types = (
    TransitiveMorphismType(
        "∇_{e,0}",
        PrimordialIntersection({PrimordialClass.SURJ_E}),
        PrimordialVacuousClass.ID_EMPTY,
    ),
    TransitiveMorphismType(
        "∇_{e,1}",
        PrimordialIntersection(),
        PrimordialVacuousClass.INIT_E,
    ),
    TransitiveMorphismType(
        "∇_{e,>1}",
        PrimordialIntersection({PrimordialClass.INJ_E}),
        PrimordialVacuousClass.INIT_E,
    ),
    TransitiveMorphismType(
        "fixed target (n=0, m=0)",
        PrimordialIntersection({PrimordialClass.SURJ_CP}),
        PrimordialVacuousClass.ID_EMPTY_CP,
    ),
    TransitiveMorphismType(
        "(n=0, m=1)",
        PrimordialIntersection(),
        PrimordialVacuousClass.INIT_CP,
    ),
    TransitiveMorphismType(
        "(n=0, m>1)",
        PrimordialIntersection({PrimordialClass.INJ_CP}),
        PrimordialVacuousClass.INIT_CP,
    ),
    TransitiveMorphismType(
        "(n=1, m=0)",
        PrimordialIntersection({PrimordialClass.PB}),
        PrimordialVacuousClass.INIT_CP_INTERSECT_PB,
    ),
    TransitiveMorphismType(
        "(n>1, m=0)",
        PrimordialIntersection({PrimordialClass.PB, PrimordialClass.INJ_CP}),
        PrimordialVacuousClass.INIT_CP_INTERSECT_PB,
    ),
    TransitiveMorphismType(
        "(n>=1, m>=1)",
        PrimordialIntersection({PrimordialClass.HOMOG, PrimordialClass.INJ_CP}),
        PrimordialVacuousClass.INIT_CP,
    ),
)

The induced poset of *transitive right-lifting classes* is obtained directly from the ambient primordial-intersection semilattice. Closing those same derived classes under meets gives the *transitive-generated right-lifting classes*. Sage constructs both substructures in facade mode, so their elements remain the same immutable `PrimordialIntersection` values used by the ambient structure.

In [71]:
transitive_rlcs = tuple(sorted(
    {morphism_type.rlc for morphism_type in transitive_morphism_types},
    key=lambda x: x.sort_key
))

primordial_vacuous_classes = tuple(sorted(
    {morphism_type.vacuous_rlc for morphism_type in transitive_morphism_types},
    key=lambda x: x.sort_key
))

transitive_rlc_profiles = frozenset(
    (morphism_type.rlc, morphism_type.vacuous_rlc)
    for morphism_type in transitive_morphism_types
)

transitive_rlc_poset = primordial_intersection_semilattice.subposet(
    transitive_rlcs
)

transitive_generated_rlcs = primordial_intersection_semilattice.submeetsemilattice(
    transitive_rlcs
)

# Part 2. Right lifting classes of single morphisms

Any morphism $f : X \to Y$ may be written as a disjoint union of morphisms $f_i : X_i \to Y_i$ where each $Y_i$ is transitive. Then the right-lifting class of $\{f\}$ is given by $$\{f\}^\perp = \bigcap_i \{f_i\}^\perp \cup \bigcup_i \{f_i\}^\varnothing.$$
We call such classes *singleton right-lifting classes*, and more generally we call any class of the form $A \cup A'$ with $A$ a primordial intersection and $A'$ a collateral class a *mixed class*. If $\mathcal{G}(A')$ denotes the canonical set of primordial vacuous generators stored by $A'$, consider the following candidate containment relation:
$$A \cup A' \preceq B \cup B' \quad\Longleftrightarrow\quad A \subseteq B \text{ and, for every } C \in \mathcal{G}(A'),\ C \subseteq B \text{ or } C \subseteq B'.$$
Every containment declared by this relation is clearly a genuine containment of classes. To show the converse, and thus establish that ${\preceq}$ is exactly the containment relation for mixed classes, we use the following procedure:
1. We define a type `MixedClass` storing a (purportedly) canonical form for mixed classes. We verify that two pairs $(A,A'), (B,B')$ are stored as the same `MixedClass` object only if they are equal.
2. We verify that $\preceq$ is in fact a partial order on these canonicalized `MixedClass`es. Then, we generate the poset of `MixedClass`es according to $\preceq$. Recalling that we know that $\preceq$ is no stronger than true containment, this can only differ from the true poset of mixed classes by failing to record some true containments.
3. We verify that, for each non-containment recorded in this poset, we have an example morphism witnessing the non-containment of those mixed classes.

We first record a table of which primordial vacuous classes are contained in which primordial classes -- this allows us to test containments of the form $A' \subseteq B$.

In [72]:
# Table 5: the primordial classes containing each primordial vacuous class.
PRIMORDIAL_VACUOUS_PRIMORDIAL_CONTAINMENTS = {
    PrimordialVacuousClass.ID_EMPTY: frozenset(PrimordialClass),
    PrimordialVacuousClass.INIT_E: frozenset({
        PrimordialClass.INJ_E,
        PrimordialClass.INJ_CP,
        PrimordialClass.PB,
        PrimordialClass.HOMOG,
    }),
    PrimordialVacuousClass.ID_EMPTY_CP: frozenset({
        PrimordialClass.INJ_CP,
        PrimordialClass.SURJ_CP,
        PrimordialClass.PB,
        PrimordialClass.HOMOG,
    }),
    PrimordialVacuousClass.INIT_CP_INTERSECT_PB: frozenset({
        PrimordialClass.INJ_CP,
        PrimordialClass.PB,
        PrimordialClass.HOMOG,
    }),
    PrimordialVacuousClass.INIT_CP: frozenset({
        PrimordialClass.INJ_CP,
        PrimordialClass.HOMOG,
    }),
}


def primordial_vacuous_is_contained_in_primordial_intersection(
    primordial_vacuous_class,
    primordial_intersection,
):
    return (
        primordial_intersection.factors
        <= PRIMORDIAL_VACUOUS_PRIMORDIAL_CONTAINMENTS[
            primordial_vacuous_class
        ]
    )


def collateral_is_contained_in_primordial_intersection(
    collateral_class,
    primordial_intersection,
):
    """Return whether A' ⊆ A for a collateral A' and primordial intersection A."""
    return all(
        primordial_vacuous_is_contained_in_primordial_intersection(
            generator,
            primordial_intersection,
        )
        for generator in collateral_class.generators
    )


# Tables 8 and 9 witness every non-containment asserted by Table 5.
primordial_vacuous_primordial_noncontainment_witnesses = {
    (primordial_vacuous_class, primordial_class): find_noncontainment_witness(
        primordial_vacuous_class,
        primordial_class,
        example_lies_in_primordial_vacuous_class,
        example_lies_in_primordial_class
    )
    for primordial_vacuous_class in PrimordialVacuousClass
    for primordial_class in PrimordialClass
    if (
        primordial_class
        not in PRIMORDIAL_VACUOUS_PRIMORDIAL_CONTAINMENTS[
            primordial_vacuous_class
        ]
    )
}


We can now define a `MixedClass` type, which stores a primordial intersection $A$ and a collateral class $A'$, which is canonicalized upon construction of the `MixedClass`. The canonicalization is performed as follows: given $A$ a primordial intersection and $A'$ a collateral class, we remove from $A'$ every primordial vacuous generator already contained in $A$, then canonicalizing the remaining union.

For example, if $\{\operatorname{id}_\varnothing\}^{C_p} \subseteq A$, then $A \cup (\mathsf{init}^e \cup \{\operatorname{id}_\varnothing\}^{C_p}) = A \cup \mathsf{init}^e$.

This means that the canonicalization process looks like $(A, C \cup A') \rightsquigarrow (A, A')$ where $C$ is a collateral class contained in $A$; it follows that $A \cup C \cup A' = A \cup A'$. Thus, two pairs $(A,A')$, $(B,B')$ are stored as the same mixed class object only if $A \cup A' = B \cup B'$.

In [73]:
def collateral_remainder(collateral_class, primordial_intersection):
    """Remove generators already absorbed by a primordial part."""
    return CollateralClass(
        generator
        for generator in collateral_class.generators
        if not primordial_vacuous_is_contained_in_primordial_intersection(
            generator,
            primordial_intersection,
        )
    )

@dataclass(frozen=True)
class MixedClass:
    """A canonically stored mixed class A ∪ A'."""

    primordial_part: PrimordialIntersection
    collateral_part: CollateralClass

    def __post_init__(self):
        if not isinstance(self.primordial_part, PrimordialIntersection):
            raise TypeError("primordial_part must be a PrimordialIntersection")
        if not isinstance(self.collateral_part, CollateralClass):
            raise TypeError("collateral_part must be a CollateralClass")
        object.__setattr__(
            self,
            "collateral_part",
            collateral_remainder(
                self.collateral_part,
                self.primordial_part,
            ),
        )

    def __le__(self, other):
        if not isinstance(other, MixedClass):
            return NotImplemented
        return (
            self.primordial_part <= other.primordial_part
            and all(
                primordial_vacuous_is_contained_in_primordial_intersection(
                    generator,
                    other.primordial_part,
                )
                or primordial_vacuous_is_contained_in_collateral_class(
                    generator,
                    other.collateral_part,
                )
                for generator in self.collateral_part.generators
            )
        )

    def __lt__(self, other):
        if not isinstance(other, MixedClass):
            return NotImplemented
        return self != other and self <= other

    @property
    def sort_key(self):
        return (self.primordial_part.sort_key, self.collateral_part.sort_key)

    def __str__(self):
        return f"({self.primordial_part}) ∪ ({self.collateral_part})"

    def __repr__(self):
        return str(self)

    def _latex_(self):
        return (
            rf"\left({self.primordial_part._latex_()}\right) \cup "
            rf"\left({self.collateral_part.latex}\right)"
        )


We can now construct the collection of all `MixedClass` objects.

We have implemented `<=` on `MixedClass`es according to $\preceq$, and it is easy to show (but not completely obvious) that this defines a partial order. We include a mechanical verification of this fact for completeness, then construct the poset of all `MixedClass` objects.

In [74]:
# All mixed classes.
mixed_classes = tuple(sorted(
    {
        MixedClass(primordial_part, collateral_part)
        for primordial_part in primordial_intersections
        for collateral_part in collateral_classes
    },
    key=lambda mixed_class: mixed_class.sort_key,
))

mixed_class_preceq_graph = DiGraph(
    (
        mixed_classes,
        [
            (left, right)
            for left in mixed_classes
            for right in mixed_classes
            if left != right and left <= right
        ],
    ),
    format="vertices_and_edges",
)

assert mixed_class_preceq_graph.antisymmetric()
assert mixed_class_preceq_graph.is_transitive()

mixed_class_poset = Poset(
    (mixed_classes, lambda left, right: left <= right),
    facade=True,
)

Finally, we verify that each non-containment in `mixed_class_poset` is correct. This shows that `mixed_class_poset` is indeed the poset of mixed classes, under containment.

In [75]:
def example_lies_in_mixed_class(example, mixed_class):
    return (
        example_lies_in_primordial_intersection(
            example,
            mixed_class.primordial_part,
        )
        or example_lies_in_collateral_class(
            example,
            mixed_class.collateral_part,
        )
    )


# Every declared comparison is visibly a genuine containment. Conversely,
# Tables 8 and 9 witness every comparison omitted by the candidate order.
mixed_noncontainment_witnesses = {
    (left, right): find_noncontainment_witness(
        left,
        right,
        example_lies_in_mixed_class,
        example_lies_in_mixed_class
    )
    for left in mixed_classes
    for right in mixed_classes
    if not mixed_class_poset.le(left, right)
}

We can now compute the poset of all right-lifting classes of single morphisms -- it is simply the subposet of `mixed_class_poset` consisting of mixed classes of the form $\bigcap_i \{f_i\}^\perp \cup \bigcup_i \{f_i\}^\varnothing$ for some collection $\{f_i\}_i$ of morphisms with transitive codomain.

In [76]:
def singleton_rlc_from_transitive_types(morphism_types):
    """Compute the RLC of the coproduct of the chosen transitive morphism types."""
    primordial_part = primordial_intersection_semilattice.top()
    collateral_part = collateral_semilattice.bottom()

    for morphism_type in morphism_types:
        primordial_part = primordial_intersection_semilattice.meet(
            primordial_part,
            morphism_type.rlc,
        )
        collateral_part = collateral_semilattice.join(
            collateral_part,
            CollateralClass({morphism_type.vacuous_rlc}),
        )

    return MixedClass(primordial_part, collateral_part)

singleton_rlcs = tuple(sorted(
    {
        mixed_class_poset(singleton_rlc_from_transitive_types(selection))
        for selection in subsets(transitive_morphism_types)
    },
    key=lambda mixed_class: mixed_class.sort_key,
))

singleton_rlc_poset = mixed_class_poset.subposet(singleton_rlcs)

# Part 3. Arbitrary right-lifting classes.

We first close the collateral classes under intersections with one another and with primordial classes. We call the resulting thirteen classes from Table 7 *collateral intersections*. Each is an explicit member of the `CollateralIntersection` enum and stores a canonical pair representing
$$P \cap \bigcap_{C \in \mathcal C} C,$$
where $P$ is a `PrimordialIntersection` and $\mathcal C$ is a nonempty set of `CollateralClass`es. The class method `CollateralIntersection.from_factors(P, C)` uses Table 7 to return the enum member represented by any such pair.

In [77]:
class CollateralIntersection(Enum):
    """One of the thirteen canonical collateral intersections in Table 7."""

    INIT_CP = (
        PrimordialIntersection(),
        frozenset({
            CollateralClass({PrimordialVacuousClass.INIT_CP}),
        }),
    )
    INIT_CP_INTERSECT_PB = (
        PrimordialIntersection(),
        frozenset({
            CollateralClass({
                PrimordialVacuousClass.INIT_CP_INTERSECT_PB,
            }),
        }),
    )
    INIT_E_UNION_ID_EMPTY_CP = (
        PrimordialIntersection(),
        frozenset({
            CollateralClass({
                PrimordialVacuousClass.INIT_E,
                PrimordialVacuousClass.ID_EMPTY_CP,
            }),
        }),
    )
    INIT_E = (
        PrimordialIntersection(),
        frozenset({
            CollateralClass({PrimordialVacuousClass.INIT_E}),
        }),
    )
    ID_EMPTY_CP = (
        PrimordialIntersection(),
        frozenset({
            CollateralClass({PrimordialVacuousClass.ID_EMPTY_CP}),
        }),
    )
    ID_EMPTY_CP_INTERSECT_INIT_E = (
        PrimordialIntersection(),
        frozenset({
            CollateralClass({PrimordialVacuousClass.ID_EMPTY_CP}),
            CollateralClass({PrimordialVacuousClass.INIT_E}),
        }),
    )
    ID_EMPTY = (
        PrimordialIntersection(),
        frozenset({
            CollateralClass({PrimordialVacuousClass.ID_EMPTY}),
        }),
    )
    INIT_CP_INTERSECT_INJ_E = (
        PrimordialIntersection({PrimordialClass.INJ_E}),
        frozenset({
            CollateralClass({PrimordialVacuousClass.INIT_CP}),
        }),
    )
    INIT_E_UNION_ID_EMPTY_CP_INTERSECT_INJ_E = (
        PrimordialIntersection({PrimordialClass.INJ_E}),
        frozenset({
            CollateralClass({
                PrimordialVacuousClass.INIT_E,
                PrimordialVacuousClass.ID_EMPTY_CP,
            }),
        }),
    )
    INIT_CP_INTERSECT_SURJ_E = (
        PrimordialIntersection({PrimordialClass.SURJ_E}),
        frozenset({
            CollateralClass({PrimordialVacuousClass.INIT_CP}),
        }),
    )
    ID_EMPTY_CP_INTERSECT_INJ_E = (
        PrimordialIntersection({PrimordialClass.INJ_E}),
        frozenset({
            CollateralClass({PrimordialVacuousClass.ID_EMPTY_CP}),
        }),
    )
    ID_EMPTY_CP_INTERSECT_SURJ_E = (
        PrimordialIntersection({PrimordialClass.SURJ_E}),
        frozenset({
            CollateralClass({PrimordialVacuousClass.ID_EMPTY_CP}),
        }),
    )
    ID_EMPTY_CP_INTERSECT_BIJ_E = (
        PrimordialIntersection({
            PrimordialClass.INJ_E,
            PrimordialClass.SURJ_E,
        }),
        frozenset({
            CollateralClass({PrimordialVacuousClass.ID_EMPTY_CP}),
        }),
    )

    @property
    def primordial_part(self):
        return self.value[0]

    @property
    def collateral_factors(self):
        return self.value[1]

    def _ordered_collateral_factors(self):
        return tuple(sorted(
            self.collateral_factors,
            key=lambda factor: factor.sort_key,
        ))

    @property
    def sort_key(self):
        """Derive a deterministic ordering from the two stored components."""
        return (
            self.primordial_part.sort_key,
            tuple(
                factor.sort_key
                for factor in self._ordered_collateral_factors()
            ),
        )

    @classmethod
    def from_factors(cls, primordial_part, collateral_factors):
        """Return the Table 7 member represented by the supplied factors."""
        if not isinstance(primordial_part, PrimordialIntersection):
            raise TypeError(
                "primordial_part must be a PrimordialIntersection"
            )

        collateral_factors = frozenset(collateral_factors)
        if not collateral_factors:
            raise ValueError("collateral_factors must be nonempty")
        if not all(
            isinstance(factor, CollateralClass)
            for factor in collateral_factors
        ):
            raise TypeError(
                "collateral_factors must contain CollateralClass values"
            )

        # Remove every collateral factor which contains another factor, since
        # these don't affect the intersection of collateral factors.
        minimal_factors = frozenset(
            factor
            for factor in collateral_factors
            if not any(
                other != factor and other <= factor
                for other in collateral_factors
            )
        )

        # Look for a Table 7 member with no primordial factors and the same minimal
        # set of collateral factors. If none exists, there is a problem with Table 7.
        try:
            presentation = next(
                member
                for member in cls
                if not member.primordial_part.factors
                and member.collateral_factors == minimal_factors
            )
        except StopIteration:
            raise AssertionError(
                "Table 7 does not contain this intersection of collateral classes"
            )

        # Now successively intersect the primordial factors into our representative.
        # Note that `TABLE_7_PRIMORDIAL_INTERSECTIONS` is defined after the CollateralIntersection class.
        for primordial_class in primordial_part.factors:
            presentation = TABLE_7_PRIMORDIAL_INTERSECTIONS[presentation][primordial_class]

        return presentation

    def intersection(self, other):
        if not isinstance(other, CollateralIntersection):
            return NotImplemented
        return type(self).from_factors(
            self.primordial_part.intersection(other.primordial_part),
            self.collateral_factors | other.collateral_factors,
        )

    def __le__(self, other):
        if not isinstance(other, CollateralIntersection):
            return NotImplemented
        return self.intersection(other) is self

    def __lt__(self, other):
        if not isinstance(other, CollateralIntersection):
            return NotImplemented
        return self is not other and self <= other

    def __str__(self):
        has_other_factors = (
            len(self.collateral_factors) > 1
            or bool(self.primordial_part.factors)
        )
        parts = [
            (
                f"({factor})"
                if len(factor.generators) > 1 and has_other_factors
                else str(factor)
            )
            for factor in self._ordered_collateral_factors()
        ]
        if self.primordial_part.factors:
            parts.append(str(self.primordial_part))
        return " ∩ ".join(parts)

    def __repr__(self):
        return str(self)

    def _latex_string(self):
        has_other_factors = (
            len(self.collateral_factors) > 1
            or bool(self.primordial_part.factors)
        )
        parts = [
            (
                rf"\left({factor._latex_()}\right)"
                if len(factor.generators) > 1 and has_other_factors
                else factor._latex_()
            )
            for factor in self._ordered_collateral_factors()
        ]
        if self.primordial_part.factors:
            parts.append(self.primordial_part._latex_())
        return r" \cap ".join(parts)


# Python 3.14 reserves _sunder_ names inside Enum definitions, so install
# Sage's customary LaTeX hook after the enum class has been created.
CollateralIntersection._latex_ = CollateralIntersection._latex_string


# Table 7: intersect a canonical row with one primordial class.
TABLE_7_PRIMORDIAL_INTERSECTIONS = {
    CollateralIntersection.INIT_CP: {
        PrimordialClass.INJ_E: CollateralIntersection.INIT_CP_INTERSECT_INJ_E,
        PrimordialClass.INJ_CP: CollateralIntersection.INIT_CP,
        PrimordialClass.SURJ_E: CollateralIntersection.INIT_CP_INTERSECT_SURJ_E,
        PrimordialClass.SURJ_CP: CollateralIntersection.ID_EMPTY_CP,
        PrimordialClass.PB: CollateralIntersection.INIT_CP_INTERSECT_PB,
        PrimordialClass.HOMOG: CollateralIntersection.INIT_CP,
    },
    CollateralIntersection.INIT_CP_INTERSECT_PB: {
        PrimordialClass.INJ_E: CollateralIntersection.INIT_CP_INTERSECT_INJ_E,
        PrimordialClass.INJ_CP: CollateralIntersection.INIT_CP_INTERSECT_PB,
        PrimordialClass.SURJ_E: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_SURJ_E
        ),
        PrimordialClass.SURJ_CP: CollateralIntersection.ID_EMPTY_CP,
        PrimordialClass.PB: CollateralIntersection.INIT_CP_INTERSECT_PB,
        PrimordialClass.HOMOG: CollateralIntersection.INIT_CP_INTERSECT_PB,
    },
    CollateralIntersection.INIT_E_UNION_ID_EMPTY_CP: {
        PrimordialClass.INJ_E: (
            CollateralIntersection
            .INIT_E_UNION_ID_EMPTY_CP_INTERSECT_INJ_E
        ),
        PrimordialClass.INJ_CP: (
            CollateralIntersection.INIT_E_UNION_ID_EMPTY_CP
        ),
        PrimordialClass.SURJ_E: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_SURJ_E
        ),
        PrimordialClass.SURJ_CP: CollateralIntersection.ID_EMPTY_CP,
        PrimordialClass.PB: (
            CollateralIntersection.INIT_E_UNION_ID_EMPTY_CP
        ),
        PrimordialClass.HOMOG: (
            CollateralIntersection.INIT_E_UNION_ID_EMPTY_CP
        ),
    },
    CollateralIntersection.INIT_E: {
        PrimordialClass.INJ_E: CollateralIntersection.INIT_E,
        PrimordialClass.INJ_CP: CollateralIntersection.INIT_E,
        PrimordialClass.SURJ_E: CollateralIntersection.ID_EMPTY,
        PrimordialClass.SURJ_CP: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_INIT_E
        ),
        PrimordialClass.PB: CollateralIntersection.INIT_E,
        PrimordialClass.HOMOG: CollateralIntersection.INIT_E,
    },
    CollateralIntersection.ID_EMPTY_CP: {
        PrimordialClass.INJ_E: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_INJ_E
        ),
        PrimordialClass.INJ_CP: CollateralIntersection.ID_EMPTY_CP,
        PrimordialClass.SURJ_E: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_SURJ_E
        ),
        PrimordialClass.SURJ_CP: CollateralIntersection.ID_EMPTY_CP,
        PrimordialClass.PB: CollateralIntersection.ID_EMPTY_CP,
        PrimordialClass.HOMOG: CollateralIntersection.ID_EMPTY_CP,
    },
    CollateralIntersection.ID_EMPTY_CP_INTERSECT_INIT_E: {
        PrimordialClass.INJ_E: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_INIT_E
        ),
        PrimordialClass.INJ_CP: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_INIT_E
        ),
        PrimordialClass.SURJ_E: CollateralIntersection.ID_EMPTY,
        PrimordialClass.SURJ_CP: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_INIT_E
        ),
        PrimordialClass.PB: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_INIT_E
        ),
        PrimordialClass.HOMOG: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_INIT_E
        ),
    },
    CollateralIntersection.ID_EMPTY: {
        primordial_class: CollateralIntersection.ID_EMPTY
        for primordial_class in PrimordialClass
    },
    CollateralIntersection.INIT_CP_INTERSECT_INJ_E: {
        PrimordialClass.INJ_E: (
            CollateralIntersection.INIT_CP_INTERSECT_INJ_E
        ),
        PrimordialClass.INJ_CP: (
            CollateralIntersection.INIT_CP_INTERSECT_INJ_E
        ),
        PrimordialClass.SURJ_E: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_BIJ_E
        ),
        PrimordialClass.SURJ_CP: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_INJ_E
        ),
        PrimordialClass.PB: (
            CollateralIntersection.INIT_CP_INTERSECT_INJ_E
        ),
        PrimordialClass.HOMOG: (
            CollateralIntersection.INIT_CP_INTERSECT_INJ_E
        ),
    },
    CollateralIntersection.INIT_E_UNION_ID_EMPTY_CP_INTERSECT_INJ_E: {
        PrimordialClass.INJ_E: (
            CollateralIntersection
            .INIT_E_UNION_ID_EMPTY_CP_INTERSECT_INJ_E
        ),
        PrimordialClass.INJ_CP: (
            CollateralIntersection
            .INIT_E_UNION_ID_EMPTY_CP_INTERSECT_INJ_E
        ),
        PrimordialClass.SURJ_E: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_BIJ_E
        ),
        PrimordialClass.SURJ_CP: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_INJ_E
        ),
        PrimordialClass.PB: (
            CollateralIntersection
            .INIT_E_UNION_ID_EMPTY_CP_INTERSECT_INJ_E
        ),
        PrimordialClass.HOMOG: (
            CollateralIntersection
            .INIT_E_UNION_ID_EMPTY_CP_INTERSECT_INJ_E
        ),
    },
    CollateralIntersection.INIT_CP_INTERSECT_SURJ_E: {
        PrimordialClass.INJ_E: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_BIJ_E
        ),
        PrimordialClass.INJ_CP: (
            CollateralIntersection.INIT_CP_INTERSECT_SURJ_E
        ),
        PrimordialClass.SURJ_E: (
            CollateralIntersection.INIT_CP_INTERSECT_SURJ_E
        ),
        PrimordialClass.SURJ_CP: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_SURJ_E
        ),
        PrimordialClass.PB: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_SURJ_E
        ),
        PrimordialClass.HOMOG: (
            CollateralIntersection.INIT_CP_INTERSECT_SURJ_E
        ),
    },
    CollateralIntersection.ID_EMPTY_CP_INTERSECT_INJ_E: {
        PrimordialClass.INJ_E: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_INJ_E
        ),
        PrimordialClass.INJ_CP: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_INJ_E
        ),
        PrimordialClass.SURJ_E: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_BIJ_E
        ),
        PrimordialClass.SURJ_CP: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_INJ_E
        ),
        PrimordialClass.PB: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_INJ_E
        ),
        PrimordialClass.HOMOG: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_INJ_E
        ),
    },
    CollateralIntersection.ID_EMPTY_CP_INTERSECT_SURJ_E: {
        PrimordialClass.INJ_E: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_BIJ_E
        ),
        PrimordialClass.INJ_CP: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_SURJ_E
        ),
        PrimordialClass.SURJ_E: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_SURJ_E
        ),
        PrimordialClass.SURJ_CP: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_SURJ_E
        ),
        PrimordialClass.PB: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_SURJ_E
        ),
        PrimordialClass.HOMOG: (
            CollateralIntersection.ID_EMPTY_CP_INTERSECT_SURJ_E
        ),
    },
    CollateralIntersection.ID_EMPTY_CP_INTERSECT_BIJ_E: {
        primordial_class: CollateralIntersection.ID_EMPTY_CP_INTERSECT_BIJ_E
        for primordial_class in PrimordialClass
    },
}

Now we can produce the poset of all collateral intersections, which is a meet-semilattice under intersection. We implement containment by $x \leq y$ iff $x \cap y = x$. Assuming the intersection computations in Table 7 are correct, this is exactly containment of the corresponding mathematical classes.

It remains to verify that the thirteen enum members represent distinct classes. For every pair, we therefore find an example morphism from Tables 8 and 9 which lies in exactly one of the two classes.

In [78]:
collateral_intersections = tuple(sorted(
    CollateralIntersection,
    key=lambda x: x.sort_key,
))

def example_lies_in_collateral_intersection(
    example,
    collateral_intersection,
):
    return (
        example_lies_in_primordial_intersection(
            example,
            collateral_intersection.primordial_part,
        )
        and all(
            example_lies_in_collateral_class(example, factor)
            for factor in collateral_intersection.collateral_factors
        )
    )

def find_collateral_intersection_distinguishing_example(left, right):
    """Return an example lying in exactly one of the two classes."""
    for example in example_morphisms:
        if (
            example_lies_in_collateral_intersection(example, left)
            != example_lies_in_collateral_intersection(example, right)
        ):
            return example
    raise AssertionError(
        f"no example distinguishes {left} from {right}"
    )

collateral_intersection_distinguishing_examples = {
    (left, right): find_collateral_intersection_distinguishing_example(
        left,
        right,
    )
    for index, left in enumerate(collateral_intersections)
    for right in collateral_intersections[index + 1:]
}

collateral_intersection_semilattice = MeetSemilattice(
    (collateral_intersections, lambda left, right: left <= right),
    facade=True,
)

Next, we turn to unions of collateral intersections. Because the poset of collateral intersections happens to be a lattice, the lattice join $\bigvee_i Q_i$ is the only possible collateral intersection which could equal the literal union $\bigcup_i Q_i$. The literal union may nevertheless be strictly smaller than that join.

Besides the tautological singleton equalities, exactly three such unions are collateral intersections. Write $E=\mathsf{init}^e$, $I=\{\mathrm{id}_\varnothing\}^{C_p}$, $U=E\cup I$, and $J=\mathsf{inj}^e$. They are

$$E\cup I=U,$$

which is the definition of $U$;

$$E\cup(I\cap J)=U\cap J,$$

because $E\subseteq J$ and intersection distributes over union; and

$$I\cup(U\cap J)=U,$$

because distributivity gives $I\cup(U\cap J)=U\cap(I\cup J)$, while $U=E\cup I\subseteq I\cup J$.

To show that there are no other equalities, it suffices to consider antichains of at least two collateral intersections: comparable summands are redundant, while singleton antichains give tautological equalities. For every antichain other than the three above, we compute its lattice join and find an example from Tables 8 and 9 which lies in the join but outside every summand. Such an example proves that the literal union is not a collateral intersection.

In [79]:
PROVEN_COLLATERAL_INTERSECTION_UNIONS = {
    frozenset({
        CollateralIntersection.INIT_E,
        CollateralIntersection.ID_EMPTY_CP,
    }): CollateralIntersection.INIT_E_UNION_ID_EMPTY_CP,
    frozenset({
        CollateralIntersection.INIT_E,
        CollateralIntersection.ID_EMPTY_CP_INTERSECT_INJ_E,
    }): (
        CollateralIntersection
        .INIT_E_UNION_ID_EMPTY_CP_INTERSECT_INJ_E
    ),
    frozenset({
        CollateralIntersection.ID_EMPTY_CP,
        (
            CollateralIntersection
            .INIT_E_UNION_ID_EMPTY_CP_INTERSECT_INJ_E
        ),
    }): CollateralIntersection.INIT_E_UNION_ID_EMPTY_CP,
}


def maximal_collateral_intersection_summands(summands):
    """Remove duplicate summands and summands contained in another."""
    summands = frozenset(summands)
    return frozenset(
        summand
        for summand in summands
        if not any(summand < other for other in summands)
    )


def union_equals_collateral_intersection(summands, candidate):
    """Test whether a finite union is the given collateral intersection."""
    maximal_summands = maximal_collateral_intersection_summands(
        summands
    )
    if len(maximal_summands) == 1:
        return next(iter(maximal_summands)) is candidate
    return (
        PROVEN_COLLATERAL_INTERSECTION_UNIONS.get(maximal_summands)
        is candidate
    )


def collateral_intersection_is_contained_in_union(source, targets):
    """Test source ⊆ ⋃ targets using source = ⋃(source ∩ target)."""
    return union_equals_collateral_intersection(
        (source.intersection(target) for target in targets),
        source,
    )


def intersect_collateral_intersection_with_primordial(
    collateral_intersection,
    primordial_intersection,
):
    return CollateralIntersection.from_factors(
        collateral_intersection.primordial_part.intersection(
            primordial_intersection
        ),
        collateral_intersection.collateral_factors,
    )


def collateral_intersection_is_contained_in_primordial_union(
    source,
    primordial_summand,
    collateral_summands,
):
    """Test source ⊆ primordial_summand ∪ ⋃ collateral_summands."""
    pieces = [
        intersect_collateral_intersection_with_primordial(
            source,
            primordial_summand,
        )
    ]
    pieces.extend(
        source.intersection(summand)
        for summand in collateral_summands
    )
    return union_equals_collateral_intersection(pieces, source)


def join_collateral_intersections(intersections):
    """Return the lattice join of a nonempty collection."""
    intersections = tuple(intersections)
    if not intersections:
        raise ValueError("the collection must be nonempty")

    result = intersections[0]
    for intersection in intersections[1:]:
        result = collateral_intersection_semilattice.join(
            result,
            intersection,
        )
    return result


def is_collateral_intersection_antichain(intersections):
    return all(
        not (left <= right or right <= left)
        for left, right in combinations(intersections, 2)
    )


def example_lies_in_collateral_intersection_union(example, summands):
    return any(
        example_lies_in_collateral_intersection(example, summand)
        for summand in summands
    )


def find_collateral_intersection_union_witness(summands, candidate_join):
    """Find an example proving that the literal union is below its join."""
    for example in example_morphisms:
        if (
            example_lies_in_collateral_intersection(
                example,
                candidate_join,
            )
            and not example_lies_in_collateral_intersection_union(
                example,
                summands,
            )
        ):
            return example
    return None


collateral_intersection_union_antichains = tuple(
    summands
    for size in range(2, len(collateral_intersections) + 1)
    for summands in combinations(collateral_intersections, size)
    if is_collateral_intersection_antichain(summands)
)

# Every nontrivial equality other than the three proved above is false.
collateral_intersection_union_non_equality_witnesses = {
    summands: (candidate_join, witness)
    for summands in collateral_intersection_union_antichains
    if frozenset(summands)
    not in PROVEN_COLLATERAL_INTERSECTION_UNIONS
    for candidate_join in (join_collateral_intersections(summands),)
    for witness in (find_collateral_intersection_union_witness(
        summands,
        candidate_join,
    ),)
    if witness is not None
}

We are now ready to consider all classes of the form
$$P \cup \bigcup_i Q_i,$$
where $P$ is a primordial intersection and the $Q_i$ are collateral intersections. We call these *mixed intersections*. We use the candidate containment relation
$$P \cup \bigcup_i Q_i \preceq R \cup \bigcup_j S_j
\quad\Longleftrightarrow\quad
P \subseteq R
\text{ and }
Q_i \subseteq R \cup \bigcup_j S_j
\text{ for every }i.$$
Every comparison declared by $\preceq$ is a genuine containment. We will verify below that the converse also holds.

To obtain a canonical presentation, fix $P$ and first find all thirteen collateral intersections contained in the represented class. We discard those already contained in $P$, then retain the maximal remaining collateral intersections. This does not change the represented class: every original summand is either absorbed by $P$ or contained in a retained maximal summand, while every retained summand is contained in the original presentation. Taking the entire containment profile before selecting its maximal elements also accounts automatically for the three exceptional union equalities above.

In [80]:
def canonical_collateral_intersection_summands(
    primordial_part,
    collateral_summands,
):
    """Return the canonical collateral summands modulo primordial_part."""
    collateral_summands = frozenset(collateral_summands)
    if not all(
        isinstance(summand, CollateralIntersection)
        for summand in collateral_summands
    ):
        raise TypeError(
            "collateral_summands must contain "
            "CollateralIntersection values"
        )

    contained_summands = (
        candidate
        for candidate in collateral_intersections
        if collateral_intersection_is_contained_in_primordial_union(
            candidate,
            primordial_part,
            collateral_summands,
        )
        and not collateral_intersection_is_contained_in_primordial_union(
            candidate,
            primordial_part,
            (),
        )
    )
    return maximal_collateral_intersection_summands(
        contained_summands
    )


@dataclass(frozen=True)
class MixedIntersection:
    """A canonical class P ∪ ⋃ Q_i."""

    primordial_part: PrimordialIntersection
    collateral_summands: frozenset = field(default_factory=frozenset)

    def __post_init__(self):
        if not isinstance(self.primordial_part, PrimordialIntersection):
            raise TypeError(
                "primordial_part must be a PrimordialIntersection"
            )
        object.__setattr__(
            self,
            "collateral_summands",
            canonical_collateral_intersection_summands(
                self.primordial_part,
                self.collateral_summands,
            ),
        )

    def _ordered_collateral_summands(self):
        return tuple(sorted(
            self.collateral_summands,
            key=lambda summand: summand.sort_key,
        ))

    def __le__(self, other):
        if not isinstance(other, MixedIntersection):
            return NotImplemented
        return (
            self.primordial_part <= other.primordial_part
            and all(
                collateral_intersection_is_contained_in_primordial_union(
                    summand,
                    other.primordial_part,
                    other.collateral_summands,
                )
                for summand in self.collateral_summands
            )
        )

    def __lt__(self, other):
        if not isinstance(other, MixedIntersection):
            return NotImplemented
        return self != other and self <= other

    @property
    def sort_key(self):
        return (
            self.primordial_part.sort_key,
            tuple(
                summand.sort_key
                for summand in self._ordered_collateral_summands()
            ),
        )

    def __str__(self):
        parts = [str(self.primordial_part)]
        parts.extend(
            f"({summand})"
            for summand in self._ordered_collateral_summands()
        )
        return " ∪ ".join(parts)

    def __repr__(self):
        return str(self)

    def _latex_(self):
        parts = [self.primordial_part._latex_()]
        parts.extend(
            rf"\left({summand._latex_()}\right)"
            for summand in self._ordered_collateral_summands()
        )
        return r" \cup ".join(parts)


# Every finite set of collateral intersections has an antichain with the
# same union, so these presentations suffice to generate every object.
all_collateral_intersection_antichains = (
    ((),)
    + tuple((summand,) for summand in collateral_intersections)
    + collateral_intersection_union_antichains
)

mixed_intersections = tuple(sorted(
    {
        MixedIntersection(primordial_part, collateral_summands)
        for primordial_part in primordial_intersections
        for collateral_summands
        in all_collateral_intersection_antichains
    },
    key=lambda mixed_intersection: mixed_intersection.sort_key,
))

# Check that the proposed relation is a partial order before asking Sage
# to construct a poset from it.

mixed_intersection_preceq_graph = DiGraph(
    (
        mixed_intersections,
        [
            (left, right)
            for left in mixed_intersections
            for right in mixed_intersections
            if left != right and left <= right
        ],
    ),
    format="vertices_and_edges",
)

assert mixed_intersection_preceq_graph.antisymmetric()
assert mixed_intersection_preceq_graph.is_transitive()

mixed_intersection_semilattice = MeetSemilattice(
    (
        mixed_intersections,
        lambda left, right: left <= right,
    ),
    facade=True,
)

Every comparison declared by $\preceq$ is a genuine containment, so the candidate poset can only omit genuine containments. We now verify every claimed non-containment directly: for each ordered pair $X \not\preceq Y$, we find an example morphism from Tables 8 and 9 which belongs to $X$ but not to $Y$. Consequently, $\preceq$ is exactly genuine containment and `mixed_intersection_poset` is the poset of all mixed intersections.

In [81]:
def example_lies_in_mixed_intersection(example, mixed_intersection):
    return (
        example_lies_in_primordial_intersection(
            example,
            mixed_intersection.primordial_part,
        )
        or any(
            example_lies_in_collateral_intersection(
                example,
                summand,
            )
            for summand in mixed_intersection.collateral_summands
        )
    )


mixed_intersection_noncontainment_witnesses = {
    (left, right): find_noncontainment_witness(
        left,
        right,
        example_lies_in_mixed_intersection,
        example_lies_in_mixed_intersection,
    )
    for left in mixed_intersections
    for right in mixed_intersections
    if not mixed_intersection_semilattice.le(left, right)
}

Finally, we compute the poset of all right-lifting classes. First we convert each `MixedClass` in `singleton_rlcs` to the corresponding canonical `MixedIntersection`. We then take the meet-subsemilattice which these elements generate inside `mixed_intersection_poset`.

This generated meet-semilattice contains the top class $\mathsf{all}$. Consequently it is a lattice: for any two elements, the set of their common upper bounds is nonempty, and the meet of all common upper bounds is their join. We let Sage verify this and convert the generated subsemilattice to a `LatticePoset`.

In [82]:
def mixed_intersection_from_mixed_class(mixed_class):
    """Convert P ∪ C from Part 2 to the canonical Part 3 type."""
    if not isinstance(mixed_class, MixedClass):
        raise TypeError("mixed_class must be a MixedClass")

    collateral_summand = CollateralIntersection.from_factors(
        PrimordialIntersection(),
        {mixed_class.collateral_part},
    )
    return MixedIntersection(
        mixed_class.primordial_part,
        {collateral_summand},
    )


singleton_rlc_mixed_intersections = tuple(sorted(
    {
        mixed_intersection_from_mixed_class(singleton_rlc)
        for singleton_rlc in singleton_rlcs
    },
    key=lambda mixed_intersection: mixed_intersection.sort_key,
))

all_rlc_meet_semilattice = mixed_intersection_semilattice.submeetsemilattice(
        singleton_rlc_mixed_intersections
    )

all_rlc_lattice = LatticePoset(all_rlc_meet_semilattice)

A premodel structure on the category of $C_p$-sets is nothing more than a pair $(x,y)$ in `all_rlc_lattice` with $x \leq y$. We compute that the number of such pairs is $1777$.

In [87]:
len(all_rlc_lattice.relations())

1777

In [89]:
# Structural checks.
assert len(primordial_intersections) == len(primordial_intersection_semilattice) == 21
assert len(PrimordialVacuousClass) == len(primordial_vacuous_poset) == 5
assert len(collateral_classes) == len(collateral_semilattice) == 6
assert len(transitive_morphism_types) == len(transitive_rlc_profiles) == 9
assert len(transitive_rlcs) == len(set(transitive_rlcs)) == len(transitive_rlc_poset) == 8
assert (
    len(primordial_vacuous_classes)
    == len(set(primordial_vacuous_classes))
    == 5
)
assert set(primordial_vacuous_classes) == set(PrimordialVacuousClass)
assert len(transitive_generated_rlcs) == 19
assert len(example_morphisms) == 16
assert len(mixed_classes) == len(set(mixed_classes)) == len(mixed_class_poset) == 88
assert len(mixed_noncontainment_witnesses) == 6211
assert len(singleton_rlcs) == len(set(singleton_rlcs)) == len(singleton_rlc_poset) == 38
assert not singleton_rlc_poset.is_meet_semilattice()
assert (
    len(collateral_intersections)
    == len(set(collateral_intersections))
    == len(collateral_intersection_semilattice)
    == 13
)
assert len(collateral_intersection_distinguishing_examples) == 78
assert len(collateral_intersection_union_antichains) == 26
assert len(all_collateral_intersection_antichains) == 40
assert (
    len(collateral_intersection_union_non_equality_witnesses)
    == len(collateral_intersection_union_antichains)
    - len(PROVEN_COLLATERAL_INTERSECTION_UNIONS)
    == 23
)
assert (
    {
        frozenset(summands)
        for summands
        in collateral_intersection_union_non_equality_witnesses
    }
    | set(PROVEN_COLLATERAL_INTERSECTION_UNIONS)
    == {
        frozenset(summands)
        for summands in collateral_intersection_union_antichains
    }
)
assert all(
    join_collateral_intersections(summands) is result
    and union_equals_collateral_intersection(summands, result)
    and collateral_intersection_is_contained_in_union(
        result,
        summands,
    )
    for summands, result
    in PROVEN_COLLATERAL_INTERSECTION_UNIONS.items()
)
assert all(
    not collateral_intersection_is_contained_in_union(
        candidate_join,
        summands,
    )
    for summands, (candidate_join, witness)
    in collateral_intersection_union_non_equality_witnesses.items()
)
assert all(
    collateral_intersection_is_contained_in_union(source, (target,))
    == (source <= target)
    for source in collateral_intersections
    for target in collateral_intersections
)
assert all(
    collateral_intersection_is_contained_in_primordial_union(
        source,
        primordial_summand,
        (),
    )
    == (
        intersect_collateral_intersection_with_primordial(
            source,
            primordial_summand,
        )
        is source
    )
    for source in collateral_intersections
    for primordial_summand in primordial_intersections
)
assert (
    len(mixed_intersections)
    == len(set(mixed_intersections))
    == len(mixed_intersection_semilattice)
    == 227
)
assert len(mixed_intersection_noncontainment_witnesses) == 42287
assert (
    len(singleton_rlc_mixed_intersections)
    == len(set(singleton_rlc_mixed_intersections))
    == len(singleton_rlcs)
    == 38
)
assert len(all_rlc_meet_semilattice) == len(all_rlc_lattice) == 89
assert (
    all_rlc_lattice.top()
    == MixedIntersection(PrimordialIntersection())
)
assert all(
    example_lies_in_mixed_class(example, singleton_rlc)
    == example_lies_in_mixed_intersection(
        example,
        mixed_intersection_from_mixed_class(singleton_rlc),
    )
    for example in example_morphisms
    for singleton_rlc in singleton_rlcs
)
assert all(
    singleton_rlc_poset.le(left, right)
    == all_rlc_lattice.le(
        mixed_intersection_from_mixed_class(left),
        mixed_intersection_from_mixed_class(right),
    )
    for left in singleton_rlcs
    for right in singleton_rlcs
)

assert PrimordialIntersection({PrimordialClass.INJ_E}) == PrimordialIntersection(
    {
        PrimordialClass.INJ_E,
        PrimordialClass.INJ_CP,
        PrimordialClass.PB,
        PrimordialClass.HOMOG,
    }
)
assert PrimordialIntersection(
    {PrimordialClass.PB, PrimordialClass.INJ_CP}
) <= PrimordialIntersection({PrimordialClass.PB})

assert {
    tuple(relation) for relation in primordial_vacuous_poset.cover_relations()
} == set(PRIMORDIAL_VACUOUS_COVER_RELATIONS)
assert CollateralClass({
    PrimordialVacuousClass.INIT_E,
    PrimordialVacuousClass.INIT_CP_INTERSECT_PB,
}) == CollateralClass({PrimordialVacuousClass.INIT_CP_INTERSECT_PB})
assert all(
    collateral_semilattice.join(left, right) == left.union(right)
    for left in collateral_classes
    for right in collateral_classes
)

for structure in (
    primordial_intersection_semilattice,
    primordial_vacuous_poset,
    transitive_rlc_poset,
    transitive_generated_rlcs,
    collateral_semilattice,
    mixed_class_poset,
    singleton_rlc_poset,
    collateral_intersection_semilattice,
    mixed_intersection_semilattice,
    all_rlc_meet_semilattice,
    all_rlc_lattice,
):
    assert structure._is_facade
    assert all(structure(element) is element for element in structure)

# The induced structures reuse the ambient semilattice's actual value objects.
assert all(
    any(element is ambient_element for ambient_element in primordial_intersections)
    for element in transitive_rlc_poset
)
assert all(
    any(element is ambient_element for ambient_element in primordial_intersections)
    for element in transitive_generated_rlcs
)

# Every binary meet in the generated subsemilattice is the ambient meet.
assert all(
    transitive_generated_rlcs.meet(left, right)
    is primordial_intersection_semilattice.meet(left, right)
    for left in transitive_generated_rlcs
    for right in transitive_generated_rlcs
)

# The generated RLC lattice reuses ambient mixed-intersection values, and
# its meet operation is the restriction of the ambient meet.
assert all(
    any(element is ambient_element for ambient_element in mixed_intersections)
    for element in all_rlc_lattice
)
assert all(
    all_rlc_lattice.meet(left, right)
    is mixed_intersection_semilattice.meet(left, right)
    for left in all_rlc_lattice
    for right in all_rlc_lattice
)

# Table 7 contains every enum member and every primordial column.
assert set(collateral_intersections) == set(CollateralIntersection)
assert set(TABLE_7_PRIMORDIAL_INTERSECTIONS) == set(CollateralIntersection)
assert all(
    set(row) == set(PrimordialClass)
    for row in TABLE_7_PRIMORDIAL_INTERSECTIONS.values()
)

# Every possible presentation is recognized by the enum constructor.
assert {
    CollateralIntersection.from_factors(
        primordial_part,
        collateral_factors,
    )
    for primordial_part in primordial_intersections
    for size in range(1, len(collateral_classes) + 1)
    for collateral_factors in combinations(collateral_classes, size)
} == set(CollateralIntersection)

# The enum operation and Sage semilattice agree on every binary meet.
assert all(
    collateral_intersection_semilattice.meet(left, right)
    == left.intersection(right)
    for left in collateral_intersections
    for right in collateral_intersections
)
